In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Load and split
data = pd.read_csv("../data/processed/normalized_data.csv")
X = data.drop(columns=["default", "id"], errors='ignore')
y = data["default"]

# --- Step 1: Ensure at least 2 positive (default=1) samples in training set ---
# Separate positive and negative examples
X_pos = X[y == 1]
y_pos = y[y == 1]
X_neg = X[y == 0]
y_neg = y[y == 0]

# Force first 2 defaults into training set
X_pos_train = X_pos.iloc[:2]
y_pos_train = y_pos.iloc[:2]

# Split remaining negatives
X_neg_train, X_neg_test, y_neg_train, y_neg_test = train_test_split(X_neg, y_neg, test_size=0.3, random_state=42)

# Check if we have at least 2 positive samples
if sum(y_train == 1) < 2:
    # Get the only positive sample
    pos_X = X_train[y_train == 1]
    pos_y = y_train[y_train == 1]

    # Duplicate it once
    X_train = pd.concat([X_train, pos_X])
    y_train = pd.concat([y_train, pos_y])

# --- Step 2: Apply SMOTE (k_neighbors=1) ---
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Training set class counts after SMOTE:")
print(y_train_bal.value_counts())

# --- Step 3: Random Forest GridSearch ---
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_grid={
        'n_estimators': [50, 100],
        'max_depth': [3, 5, 10],
        'min_samples_split': [2, 5]
    },
    scoring='average_precision',
    cv=3,
    n_jobs=-1
)
rf_grid.fit(X_train_bal, y_train_bal)
print("Best Random Forest Params:", rf_grid.best_params_)

# --- Step 4: Logistic Regression GridSearch ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

lr_grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=1000),
    param_grid={'C': [0.01, 0.1, 1, 10]},
    scoring='average_precision',
    cv=3,
    n_jobs=-1
)
lr_grid.fit(X_train_scaled, y_train_bal)
print("📈 Best Logistic Regression Params:", lr_grid.best_params_)

# --- Step 5: Evaluate on 100% negative test set (realistic error analysis) ---
y_pred = lr_grid.best_estimator_.predict(X_test_scaled)
print("\nLogistic Regression Test Report (0-only test set):\n", classification_report(y_test, y_pred, zero_division=0))

✅ Training set class counts after SMOTE:
default
0    69999
1    69999
Name: count, dtype: int64
🌲 Best Random Forest Params: {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 50}
📈 Best Logistic Regression Params: {'C': 0.01}

🧪 Logistic Regression Test Report (0-only test set):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     30000

    accuracy                           1.00     30000
   macro avg       1.00      1.00      1.00     30000
weighted avg       1.00      1.00      1.00     30000



In [10]:
# --- Manually Test on Real Default(s) ---
default_sample = X[y == 1]
default_pred = lr_grid.best_estimator_.predict(scaler.transform(default_sample))

print("\nPredictions on real default(s):", default_pred)

# Optional: show decision scores
default_proba = lr_grid.best_estimator_.predict_proba(scaler.transform(default_sample))[:, 1]
print("Default prediction confidence scores:", default_proba)


🔍 Predictions on real default(s): [1]
🧠 Default prediction confidence scores: [0.9997708]
